<a href="https://colab.research.google.com/github/kamxsato/MIS444_Final_Project/blob/main/24104087_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import missingno as msno
import warnings
import scipy.stats as stats
from scipy.stats import (shapiro, levene, mannwhitneyu, kruskal,
                          ks_2samp, chi2_contingency, pearsonr, spearmanr)
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Data

In [5]:
df = pd.read_csv('/content/OECD_AI_WIDEF.csv', low_memory=False)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
df.head()

Shape: 193 rows × 66 columns
Memory usage: 0.46 MB


,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,REF_AREA,INDICATOR,SEX,AGE,URBANISATION,UNIT_MEASURE,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,AGG_METHOD,UNIT_TYPE,DECIMALS,DATABASE_ID,TIME_FORMAT,UNIT_MULT,OBS_STATUS,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,HKG,OECD_AI_PUBS_TOT,_T,_T,_T,NUMBER,_Z,_Z,_Z,MEAN,COUNT,2,OECD_AI,602,0,A,...,459.317,332.921,340.295,350.673,374.651,318.487,344.539,286.837,314.259,365.124,379.062,381.862,511.496,705.822,803.098,901.903,854.981,1108.869,905.181,747.402
1,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,WSM,OECD_AI_PUBS_TOT,_T,_T,_T,NUMBER,_Z,_Z,_Z,MEAN,COUNT,2,OECD_AI,602,0,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000,NaN,NaN,NaN
2,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,CHN,OECD_AI_PUBS_TOT,_T,_T,_T,NUMBER,_Z,_Z,_Z,MEAN,COUNT,2,OECD_AI,602,0,A,...,8976.694,9757.454,12075.409,13090.166,14094.853,12418.438,11099.484,10026.767,9349.249,8444.956,8291.048,9768.475,14421.968,19170.152,22558.845,29356.872,33997.407,38211.314,32983.885,21388.174
3,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,GUF,OECD_AI_PUBS_TOT,_T,_T,_T,NUMBER,_Z,_Z,_Z,MEAN,COUNT,2,OECD_AI,602,0,A,...,0.222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,0.800,NaN,1.333,NaN,NaN,NaN
4,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,NZL,OECD_AI_PUBS_TOT,_T,_T,_T,NUMBER,_Z,_Z,_Z,MEAN,COUNT,2,OECD_AI,602,0,A,...,111.433,90.714,109.748,99.834,99.614,132.636,155.728,118.358,141.487,156.586,207.419,161.771,193.858,191.842,264.092,198.167,204.773,212.216,151.491,99.793


In [6]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 66 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   STRUCTURE               193 non-null    object 
 1   STRUCTURE_ID            193 non-null    object 
 2   ACTION                  193 non-null    object 
 3   FREQ                    193 non-null    object 
 4   REF_AREA                193 non-null    object 
 5   INDICATOR               193 non-null    object 
 6   SEX                     193 non-null    object 
 7   AGE                     193 non-null    object 
 8   URBANISATION            193 non-null    object 
 9   UNIT_MEASURE            193 non-null    object 
 10  COMP_BREAKDOWN_1        193 non-null    object 
 11  COMP_BREAKDOWN_2        193 non-null    object 
 12  COMP_BREAKDOWN_3        193 non-null    object 
 13  AGG_METHOD              193 non-null    object 
 14  UNIT_TYPE               193 non-null    ob

In [7]:
overview = pd.DataFrame({
    'dtype':   df.dtypes,
    'nunique': df.nunique(),
    'nulls':   df.isnull().sum(),
    'null_%':  (df.isnull().mean()*100).round(2),
    'sample':  [df[c].dropna().iloc[0] if df[c].notna().any() else 'ALL NULL' for c in df.columns]
})
display(overview)

,dtype,nunique,nulls,null_%,sample
STRUCTURE,object,1,0,0.000,datastructure
STRUCTURE_ID,object,1,0,0.000,WB.DATA360:DS_DATA360(1.3)
ACTION,object,1,0,0.000,I
FREQ,object,1,0,0.000,A
REF_AREA,object,193,0,0.000,HKG
...,...,...,...,...,...
2021,float64,152,36,18.650,901.903
2022,float64,154,34,17.620,854.981
2023,float64,150,27,13.990,1108.869
2024,float64,152,29,15.030,905.181


In [8]:
df.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'REF_AREA', 'INDICATOR',
       'SEX', 'AGE', 'URBANISATION', 'UNIT_MEASURE', 'COMP_BREAKDOWN_1',
       'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'AGG_METHOD', 'UNIT_TYPE',
       'DECIMALS', 'DATABASE_ID', 'TIME_FORMAT', 'UNIT_MULT', 'OBS_STATUS',
       'DATA_SOURCE', 'OBS_CONF', 'FREQ_LABEL', 'REF_AREA_LABEL',
       'INDICATOR_LABEL', 'SEX_LABEL', 'AGE_LABEL', 'URBANISATION_LABEL',
       'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1_LABEL',
       'COMP_BREAKDOWN_2_LABEL', 'COMP_BREAKDOWN_3_LABEL', 'AGG_METHOD_LABEL',
       'UNIT_TYPE_LABEL', 'DECIMALS_LABEL', 'DATABASE_ID_LABEL',
       'TIME_FORMAT_LABEL', 'UNIT_MULT_LABEL', 'OBS_STATUS_LABEL',
       'OBS_CONF_LABEL', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023',
       '2024', '2025'],
      dtype='object')